In [1]:
import xml.etree.ElementTree as ET

# Replace with the path to your XML file
xml_file = 'flowmon-results.xml'

# Parse the XML file
tree = ET.parse(xml_file)
root = tree.getroot()

# Pretty-print the XML tree structure
def print_element(elem, level=0):
    indent = "  " * level
    print(f"{indent}<{elem.tag}", end="")
    for attr, value in elem.attrib.items():
        print(f' {attr}="{value}"', end="")
    print(">")
    for child in elem:
        print_element(child, level + 1)
    print(f"{indent}</{elem.tag}>")

# Display the full structure
print_element(root)


<FlowMonitor>
  <FlowStats>
    <Flow flowId="1" timeFirstTxPacket="+2e+09ns" timeFirstRxPacket="+2.00645e+09ns" timeLastTxPacket="+1.19e+10ns" timeLastRxPacket="+1.19003e+10ns" delaySum="+5.21853e+07ns" jitterSum="+2.87732e+07ns" lastDelay="+271034ns" maxDelay="+6.44614e+06ns" minDelay="+81034ns" txBytes="105200" rxBytes="105200" txPackets="100" rxPackets="100" lostPackets="0" timesForwarded="0">
      <delayHistogram nBins="7">
        <bin index="0" start="0" width="0.001" count="97">
        </bin>
        <bin index="1" start="0.001" width="0.001" count="2">
        </bin>
        <bin index="6" start="0.006" width="0.001" count="1">
        </bin>
      </delayHistogram>
      <jitterHistogram nBins="7">
        <bin index="0" start="0" width="0.001" count="95">
        </bin>
        <bin index="1" start="0.001" width="0.001" count="3">
        </bin>
        <bin index="6" start="0.006" width="0.001" count="1">
        </bin>
      </jitterHistogram>
      <packetSizeHistogram 

In [2]:
import xml.etree.ElementTree as ET
import numpy as np
import pandas as pd

# Load XML
tree = ET.parse('flowmon-results.xml')
root = tree.getroot()

# Prepare data storage
throughputs = []

# Iterate through FlowStats
for flow in root.find('FlowStats').findall('Flow'):
    flow_id = int(flow.attrib['flowId'])
    rx_bytes = int(flow.attrib['rxBytes'])
    time_first_tx = float(flow.attrib['timeFirstTxPacket'].replace('ns', '').replace('+', ''))
    time_last_rx = float(flow.attrib['timeLastRxPacket'].replace('ns', '').replace('+', ''))
    
    # Compute throughput in Mbps
    duration_ns = time_last_rx - time_first_tx
    if duration_ns > 0:
        throughput_mbps = (rx_bytes * 8) / duration_ns  # bits/ns
        throughput_mbps *= 1e3  # convert to Mbps
    else:
        throughput_mbps = 0.0  # avoid div by zero

    throughputs.append({
        'flowId': flow_id,
        'throughput_mbps': throughput_mbps
    })

# Convert to DataFrame
df = pd.DataFrame(throughputs)

# Compute stats
mean = df['throughput_mbps'].mean()
median = df['throughput_mbps'].median()
lower_quartile = df['throughput_mbps'].quantile(0.25)
upper_quartile = df['throughput_mbps'].quantile(0.75)

# Print results
print("Per-Flow Throughput (Mbps):\n", df)
print("\n--- Statistics ---")
print(f"Mean: {mean:.2f} Mbps")
print(f"Median: {median:.2f} Mbps")
print(f"25th Percentile: {lower_quartile:.2f} Mbps")
print(f"75th Percentile: {upper_quartile:.2f} Mbps")


Per-Flow Throughput (Mbps):
    flowId  throughput_mbps
0       1         0.085008
1       2         0.085007
2       3         0.085009
3       4         0.085009
4       5         0.085009

--- Statistics ---
Mean: 0.09 Mbps
Median: 0.09 Mbps
25th Percentile: 0.09 Mbps
75th Percentile: 0.09 Mbps


In [3]:
df

,flowId,throughput_mbps
0,1,0.085008
1,2,0.085007
2,3,0.085009
3,4,0.085009
4,5,0.085009


In [5]:
tree

In [6]:
import xml.etree.ElementTree as ET
import pandas as pd
import matplotlib.pyplot as plt
import os

# Load and parse the XML file
xml_file = "flowmon-results.xml"  # Change this to your actual filename
tree = ET.parse(xml_file)
root = tree.getroot()

# Create output folder
output_dir = "temp"
os.makedirs(output_dir, exist_ok=True)

# Extract FlowStats
flows_data = []
for flow in root.findall(".//FlowStats/Flow"):
    flow_id = int(flow.get("flowId"))
    delay_sum = float(flow.get("delaySum").replace("+", "").replace("ns", ""))
    jitter_sum = float(flow.get("jitterSum").replace("+", "").replace("ns", ""))
    tx_packets = int(flow.get("txPackets"))
    rx_packets = int(flow.get("rxPackets"))
    tx_bytes = int(flow.get("txBytes"))
    rx_bytes = int(flow.get("rxBytes"))
    time_first_tx = float(flow.get("timeFirstTxPacket").replace("+", "").replace("ns", ""))
    time_last_rx = float(flow.get("timeLastRxPacket").replace("+", "").replace("ns", ""))
    
    duration_ns = time_last_rx - time_first_tx if time_last_rx > time_first_tx else 1
    duration_sec = duration_ns * 1e-9
    
    avg_delay_ms = (delay_sum / rx_packets) * 1e-6 if rx_packets > 0 else 0
    avg_jitter_ms = (jitter_sum / rx_packets) * 1e-6 if rx_packets > 0 else 0
    pdr = rx_packets / tx_packets if tx_packets > 0 else 0
    throughput_kbps = (rx_bytes * 8 / duration_sec) / 1000 if duration_sec > 0 else 0

    flows_data.append({
        "FlowId": flow_id,
        "AvgDelay (ms)": avg_delay_ms,
        "AvgJitter (ms)": avg_jitter_ms,
        "PDR": pdr,
        "Throughput (kbps)": throughput_kbps
    })

# Convert to DataFrame
df = pd.DataFrame(flows_data)
df.to_csv(os.path.join(output_dir, "flow_stats.csv"), index=False)

# Plotting
def save_bar_plot(metric, ylabel, filename):
    plt.figure(figsize=(8, 5))
    plt.bar(df["FlowId"], df[metric], color="skyblue", edgecolor="black")
    plt.xlabel("Flow ID")
    plt.ylabel(ylabel)
    plt.title(f"{ylabel} per Flow")
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, filename))
    plt.close()

save_bar_plot("AvgDelay (ms)", "Average Delay (ms)", "avg_delay.png")
save_bar_plot("AvgJitter (ms)", "Average Jitter (ms)", "avg_jitter.png")
save_bar_plot("PDR", "Packet Delivery Ratio", "pdr.png")
save_bar_plot("Throughput (kbps)", "Throughput (kbps)", "throughput.png")

print(f"All plots and data saved in: {output_dir}")


All plots and data saved in: temp
